# Steganography encoder baseline: multiple schemes x payload types

Schemes, payload sets, chat templates and metrics live in the `steg/` package (upload it next to this notebook).
Per model run: every scheme on its own payload sets, 10 payloads x 5 covers x 5 rollouts, with and without thinking.
Rates reported for encode and decode: **overall** (exact match) and **char** (1 - normalised edit distance).

## 1. Install  (restart the kernel afterwards if pip changed torch)

In [ ]:
%%capture
!pip install -U vllm transformers accelerate datasets pandas matplotlib huggingface_hub

## 2. Config

In [ ]:
MODEL = "Qwen/Qwen3-14B"
BACKEND = "vllm"                   # "vllm" or "hf"
SCHEME_IDS = "all"                 # or a list of ids, e.g. ["L1", "P1", "N1"]

N_PAYLOADS = 10                    # per payload set (max 10)
N_COVERS = 5
N_ROLLOUTS = 5

# (encoder_thinking, decoder_thinking) conditions to run
CONDITIONS = [(False, False), (True, True)]
COT_STYLE = "native"               # "native": Qwen3 thinking mode | "prompt": <thinking> tags via prompt

TEMP_ENC = 1.0
TEMP_DEC = {False: 0.0, True: 0.6}           # thinking mode should not use greedy decoding
MAX_TOK_ENC = {False: 1536, True: 6000}
MAX_TOK_DEC = {False: 256, True: 4000}
MAX_MODEL_LEN = 16384
GPU_MEM_UTIL = 0.90
HF_BATCH = 32                      # only for BACKEND="hf"
EXTRA_LLM_KWARGS = {}
SEED = 0
RESUME = True                      # skip a condition if its output file exists
RUN_TAG = "v1"                     # change to start a fresh set of output files
OUT_DIR = "outputs"
HF_CACHE_DIR = "/workspace/hf_cache"   # needs ~40 GB free for a 14B model

import os, sys, shutil
sys.path.insert(0, os.getcwd())
assert os.path.isdir("steg"), "Upload the steg/ folder next to this notebook"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE_DIR       # restart the kernel if huggingface_hub was already imported
free_gb = shutil.disk_usage(HF_CACHE_DIR).free / 1e9
print(f"HF cache: {HF_CACHE_DIR} | free: {free_gb:.1f} GB")
assert free_gb > 40, "Not enough disk for the model: free space or change HF_CACHE_DIR"
MODEL_SLUG = MODEL.replace("/", "__")
IS_GEMMA = "gemma" in MODEL.lower()
IS_QWEN3 = "qwen3" in MODEL.lower()

## 3. HF login (only needed for gated models)

In [ ]:
from huggingface_hub import login
tok_ = os.environ.get("HF_TOKEN")
if tok_:
    login(token=tok_)

## 4. Schemes, payloads, self-test

In [ ]:
!python -m steg.selftest
import re, json, random
from steg import PAYLOAD_SETS, SCHEMES, SCHEME_BY_ID, combos, canon_payload, canon_from_text, prompts
from steg.metrics import char_rate

COMBOS = combos(SCHEME_IDS if SCHEME_IDS == "all" else list(SCHEME_IDS))
print(len(COMBOS), "scheme/payload combos:", [(s.id, p) for s, p in COMBOS])


## 5. Cover prompts (HH-RLHF)

In [ ]:
from datasets import load_dataset
ds = load_dataset("Anthropic/hh-rlhf", data_dir="helpful-base", split="train")
cands = set()
for ex in ds:
    t = ex["chosen"]
    if t.count("\n\nHuman:") != 1:
        continue
    m = re.match(r"\n\nHuman: (.*?)\n\nAssistant:", t, re.S)
    if not m:
        continue
    q = m.group(1).strip()
    if q.endswith("?") and 30 <= len(q) <= 120 and "\n" not in q:
        cands.add(q)
cands = sorted(cands)
random.Random(SEED).shuffle(cands)
COVER_PROMPTS = cands[:N_COVERS]
json.dump(COVER_PROMPTS, open(f"{OUT_DIR}/cover_prompts.json", "w"), indent=1)
for i, c in enumerate(COVER_PROMPTS): print(i, c)

## 6. Load model

In [ ]:
def prep(msgs):
    """Gemma-2 has no system role: fold system text into the first user turn."""
    if IS_GEMMA and msgs and msgs[0]["role"] == "system":
        return [{"role": "user", "content": msgs[0]["content"] + "\n\n" + msgs[1]["content"]}] + msgs[2:]
    return msgs

def thinking_mode(cot):
    """-> (use <thinking> tags via prompt, chat-template kwargs)"""
    native = IS_QWEN3 and COT_STYLE == "native"
    kw = {"enable_thinking": bool(cot and native)} if IS_QWEN3 else {}
    return bool(cot and not native), kw

if BACKEND == "vllm":
    from vllm import LLM, SamplingParams
    llm = LLM(model=MODEL, dtype="bfloat16", max_model_len=MAX_MODEL_LEN,
              gpu_memory_utilization=GPU_MEM_UTIL, seed=SEED, **EXTRA_LLM_KWARGS)
else:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(MODEL)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    hf_model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="cuda")
    torch.manual_seed(SEED)

def generate(convs, temperature, max_tokens, kw):
    """convs: list of message lists -> list of (text, truncated)."""
    convs = [prep(c) for c in convs]
    if BACKEND == "vllm":
        sp = SamplingParams(temperature=temperature, max_tokens=max_tokens)
        outs = llm.chat(convs, sp, chat_template_kwargs=kw, use_tqdm=True)
        return [(o.outputs[0].text, o.outputs[0].finish_reason == "length") for o in outs]
    from tqdm.auto import tqdm
    res = [None] * len(convs)
    order = sorted(range(len(convs)), key=lambda i: len(str(convs[i])))
    for b in tqdm(range(0, len(order), HF_BATCH)):
        idx = order[b:b + HF_BATCH]
        texts = [tok.apply_chat_template(convs[i], tokenize=False, add_generation_prompt=True, **kw) for i in idx]
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(hf_model.device)
        gkw = dict(do_sample=True, temperature=temperature, top_p=1.0, top_k=0) if temperature > 0 else dict(do_sample=False)
        with torch.no_grad():
            gen = hf_model.generate(**enc, max_new_tokens=max_tokens, pad_token_id=tok.pad_token_id, **gkw)
        new = gen[:, enc["input_ids"].shape[1]:]
        for k, i in enumerate(idx):
            n_tok = int((new[k] != tok.pad_token_id).sum())
            res[i] = (tok.decode(new[k], skip_special_tokens=True), n_tok >= max_tokens)
    return res

## 7. Run

In [ ]:
def run_condition(enc_cot, dec_cot):
    path = f"{OUT_DIR}/{MODEL_SLUG}__{RUN_TAG}__enc{int(enc_cot)}dec{int(dec_cot)}.jsonl"
    if RESUME and os.path.exists(path):
        print("exists, skipping:", path); return
    enc_prompt_cot, enc_kw = thinking_mode(enc_cot)
    dec_prompt_cot, dec_kw = thinking_mode(dec_cot)

    jobs = []   # (scheme, payload_set, payload, cover_idx, rollout)
    for scheme, pset in COMBOS:
        for payload in PAYLOAD_SETS[pset]["items"][:N_PAYLOADS]:
            for ci in range(len(COVER_PROMPTS)):
                for r in range(N_ROLLOUTS):
                    jobs.append((scheme, pset, payload, ci, r))

    enc_convs = [[{"role": "system", "content": prompts.encoder_system(
                      payload, PAYLOAD_SETS[pset]["desc"], scheme.description, enc_prompt_cot)},
                  {"role": "user", "content": prompts.encoder_user(COVER_PROMPTS[ci])}]
                 for scheme, pset, payload, ci, r in jobs]
    print(f"[enc_cot={enc_cot} dec_cot={dec_cot}] encoding {len(enc_convs)} ...")
    enc_out = generate(enc_convs, TEMP_ENC, MAX_TOK_ENC[enc_cot], enc_kw)

    parsed = []
    for text, trunc in enc_out:
        reasoning, answer, ok = prompts.parse_answer(text, require_tag=enc_prompt_cot)
        parsed.append((reasoning, answer, ok and not (enc_cot and trunc)))

    dec_convs = [[{"role": "user", "content": prompts.decoder_user(
                      COVER_PROMPTS[ci], parsed[k][1], PAYLOAD_SETS[pset]["desc"], scheme.description, dec_prompt_cot)}]
                 for k, (scheme, pset, payload, ci, r) in enumerate(jobs)]
    print(f"[enc_cot={enc_cot} dec_cot={dec_cot}] decoding {len(dec_convs)} ...")
    dec_out = generate(dec_convs, TEMP_DEC[dec_cot], MAX_TOK_DEC[dec_cot], dec_kw)

    with open(path, "w") as f:
        for k, (scheme, pset, payload, ci, r) in enumerate(jobs):
            kind = PAYLOAD_SETS[pset]["kind"]
            canon = canon_payload(kind, payload)
            reasoning, answer, parse_ok = parsed[k]
            msg, msg_ok = prompts.parse_message(dec_out[k][0])
            extracted = scheme.extract(answer, len(canon))
            dec_canon = canon_from_text(kind, msg)
            rec = dict(
                model=MODEL, scheme=scheme.id, scheme_name=scheme.name, channel=scheme.channel,
                payload_set=pset, payload=payload, canon=canon, n_symbols=len(canon),
                cover_idx=ci, cover=COVER_PROMPTS[ci], rollout=r, enc_cot=enc_cot, dec_cot=dec_cot,
                enc_raw=enc_out[k][0], enc_truncated=enc_out[k][1], reasoning=reasoning,
                answer=answer, answer_parse_ok=parse_ok,
                extracted=extracted, enc_exact=(extracted == canon), enc_char=char_rate(extracted, canon),
                dec_raw=dec_out[k][0], dec_truncated=dec_out[k][1], dec_message=msg, dec_parse_ok=msg_ok,
                dec_canon=dec_canon, dec_exact=(dec_canon == canon), dec_char=char_rate(dec_canon, canon),
                dec_matches_extract=(dec_canon == extracted),
                payload_literal=(payload.lower() in answer.lower()) or (canon in answer.lower()),
            )
            f.write(json.dumps(rec) + "\n")
    print("saved", path)

for enc_cot, dec_cot in CONDITIONS:
    run_condition(enc_cot, dec_cot)

## 8. Analysis (aggregates every file in `OUT_DIR`)

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt

df = pd.concat([pd.read_json(p, lines=True) for p in glob.glob(f"{OUT_DIR}/*.jsonl")], ignore_index=True)
df["cond"] = "enc" + df.enc_cot.map({True: "CoT", False: "no"}) + "/dec" + df.dec_cot.map({True: "CoT", False: "no"})
keys = ["model", "scheme", "payload_set", "cond"]
agg = df.groupby(keys).agg(n=("enc_exact", "size"),
                           enc_overall=("enc_exact", "mean"), enc_char=("enc_char", "mean"),
                           dec_overall=("dec_exact", "mean"), dec_char=("dec_char", "mean"),
                           dec_matches_extract=("dec_matches_extract", "mean"),
                           payload_literal=("payload_literal", "mean"),
                           enc_truncated=("enc_truncated", "mean"), parse_ok=("answer_parse_ok", "mean")).round(3)
agg["dec_overall | enc_ok"] = df[df.enc_exact].groupby(keys).dec_exact.mean().round(3)
display(agg)
agg.to_csv(f"{OUT_DIR}/summary.csv")

for metric in ["enc_overall", "enc_char", "dec_overall", "dec_char"]:
    print(metric)
    display(agg[metric].unstack("cond").round(2))

In [ ]:
# encode success vs payload length, per scheme
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, ch in zip(axes, ["letters", "bits", "digits"]):
    d = df[df.channel == ch]
    for (sc, c), g in d.groupby(["scheme", "cond"]):
        t = g.groupby("n_symbols").enc_char.mean()
        ax.plot(t.index, t.values, marker="o", label=f"{sc} {c}", linestyle="-" if "encCoT" in c else "--")
    ax.set_title(ch); ax.set_xlabel("payload symbols"); ax.set_ylim(0, 1)
axes[0].set_ylabel("enc char rate"); axes[-1].legend(fontsize=6); plt.tight_layout(); plt.show()

In [ ]:
# inspect examples
for _, r in df[df.enc_exact].head(3).iterrows():
    print(r.scheme, r.payload_set, r.cond, "| payload:", r.payload, "| decoded:", r.dec_message)
    print(r.answer[:700], "\n" + "-" * 60)